Importing the **tensorflow** library and the **keras preprocessing image** package **"ImageDataGenerator"**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

Because I couldn't import the .zip file directly, I uploaded it to Google Drive and mounted the Drive so I can import it here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/S37 - dataset (1).zip" -d "/content/extracted_data/"

generation of images with rescale 1./255 so we can have the numbers between 0 and 1

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                  shear_range = 0.2,
                                  zoom_range = 0.2,
                                  horizontal_flip = True)

importing data with same parameters of generation we fixed and with target_size = (64,64)
batch_size = 32 ; meaning: it's like reading 32 data, look the loss update the weights with the optimizer, and move to another 32 till finishing all the data

In [ ]:
training_set = train_datagen.flow_from_directory('/content/extracted_data/dataset/training_set',
                                                 target_size=(64, 64),
                                                 batch_size = 32,
                                                 class_mode='binary')

Found 8000 images belonging to 2 classes.


same as train set

In [ ]:
test_gen = ImageDataGenerator(rescale = 1./255)
test_set = test_gen.flow_from_directory('/content/extracted_data/dataset/test_set',
                                        target_size=(64,64),
                                        batch_size = 32,
                                        class_mode = "binary"
                                        )

Found 2000 images belonging to 2 classes.


Because the pipeline is Sequential, Keras automatically calculates the math for every layer after that

In [ ]:
cnn  = tf.keras.models.Sequential()

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters = 32,
                               kernel_size = (3,3),
                               activation = 'relu',
                               input_shape = (64,64,3)
                               ))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


we do the MaxPool2D so we can avoid overfitting

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size = (2,2), strides = 2))

Second convolution layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters = 32,
                               kernel_size = (3,3),
                               activation = 'relu'
                               ))
cnn.add(tf.keras.layers.MaxPool2D(pool_size = (2,2), strides = 2))

Flattening

the AI cannot read Conv 2D that's why we do the flattening method so he can make it 1D

In [ ]:
cnn.add(tf.keras.layers.Flatten())

the density

By declaring a **Dense** layer, you are creating **units = 128** individual artificial neurons.
Each of the **128** neurons is trying to learn a **specific** "information" of those features.

In [ ]:
cnn.add(tf.keras.layers.Dense(units = 128,
                              activation = 'relu'))

we need to do the Sigmoid function because our output is BINARY

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## **the compiling method**

**Adam (Adaptive Moment Estimation)** is a stochastic gradient descent algorithm. It computes individual adaptive learning rates for different parameters from estimates of first and second moments of the gradients.

**Loss Function (Binary Crossentropy):** The loss function calculates the divergence (error) between the model's predicted probability distribution and the actual true label distribution

**Evaluation Metric**: A quantifiable measure (like Accuracy) evaluated at the end of each epoch to monitor the model's empirical performance, separate from the gradient calculations.

In [ ]:
cnn.compile(optimizer = 'adam',
            loss = 'binary_crossentropy',
            metrics = ['accuracy'])

training the CNN

In [ ]:
cnn.fit(x= training_set, validation_data = test_set, epochs = 25 )

Epoch 1/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 40s 155ms/step - accuracy: 0.5985 - loss: 0.6587 - val_accuracy: 0.6575 - val_loss: 0.6121
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 152ms/step - accuracy: 0.6829 - loss: 0.5869 - val_accuracy: 0.6960 - val_loss: 0.6002
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 44s 174ms/step - accuracy: 0.7212 - loss: 0.5447 - val_accuracy: 0.7480 - val_loss: 0.5217
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 37s 147ms/step - accuracy: 0.7430 - loss: 0.5140 - val_accuracy: 0.7540 - val_loss: 0.5419
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 36s 144ms/step - accuracy: 0.7630 - loss: 0.4910 - val_accuracy: 0.7495 - val_loss: 0.5202
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 151ms/step - accuracy: 0.7766 - loss: 0.4671 - val_accuracy: 0.7770 - val_loss: 0.4856
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 37s 148ms/step - accuracy: 0.7738 - loss: 0.4624 - val_accuracy: 0.7650 - val_loss: 0.5006
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 35s 141ms/step - accuracy: 0.7878 - loss: 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

In [ ]:
data_pred1 = 'please_dont_cheat.jpg'  # any cat or dog photo uploaded to Colab

In [ ]:
test_image_raw = image.load_img(data_pred1, target_size = (64,64))
test_image_array = image.img_to_array(test_image_raw) / 255.0   # same rescaling as training
test_tensor = np.expand_dims(test_image_array, axis=0)

In [ ]:
result = cnn.predict(test_tensor)
probability = result[0][0]

In [ ]:
# class_indices: {'cats': 0, 'dogs': 1}
if probability >= 0.5:
    prediction = 'DOG'
else:
    prediction = 'CAT'
print(prediction)

# Display the result
plt.imshow(image.load_img(data_pred1))
plt.title(f"AI Prediction: {prediction} \n(Raw Probability: {probability:.4f})")
plt.axis('off')
plt.show()